# 09. Context and State: two different data sources

In lessons 06 and 07, we learned about `state` and `context` separately. Now, we will trace **the same flow** and check where each value comes from, who can read it, and what happens after the next call. The example uses explicitly programmed responses instead of a real LLM, so the result is reproducible. We are not simulating logging: we treat context as data that the application has already verified.

[LangChain Documentation: data sources and context engineering](https://docs.langchain.com/oss/python/langchain/context-engineering) · [LangGraph: state and runtime context](https://docs.langchain.com/oss/python/langgraph/graph-api)

In [ ]:
# The only cell with imports.
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import NotRequired, TypedDict

from langchain.agents import create_agent
from langchain.tools import ToolRuntime, tool
from langchain_core.messages import AIMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.runtime import Runtime

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.toml").exists())
LESSONS = ROOT / "notebooks" / "langchain" / "06_middlewares"
if str(LESSONS) not in sys.path:
    sys.path.insert(0, str(LESSONS))

from lab.faults import ScriptedModel


## 1. First, plain Python

`state` is data that **changes during execution**: message history, tool results, counters, task status. `context` is data that the **application provides to a specific call**: a set user ID, language, permissions, or a service handle. Simply passing the context does not append it to the prompt or the state.

The dictionaries below are just for illustration. The actual API will appear later.

In [ ]:
state = {"messages": ["What is my plan?"], "turns": 0}
context = {"viewer_id": "u-101", "language": "pl"}
state["turns"] += 1
state, context


| Question | State | Runtime context |
|---|---|---|
| Who provides the value? | user input, model, tool, or graph code | application code at `invoke(..., context=...)` |
| Can it change during execution? | yes, via returned updates | treat as read-only input |
| Can the checkpointer persist it for `thread_id`? | yes | no; provide it again during the next call |
| Does the model see it automatically? | only messages and fields explicitly included in the request | no; only when the code places the value in the prompt or tool result |
| How to read it? | `result["messages"]`, `runtime.state`, hook/node `state` argument | `runtime.context` |

**Trust boundary:** The server takes `viewer_id` and permissions from a previously verified session. User text or `state` generated by the model should not grant permissions. The checkpointer stores the thread state, it does not authorize requests.

## 2. Context is provided by the application, the tool reads the state

`RequestContext` is a small, immutable contract. A profile dictionary replaces the database. The model only knows the tool's name; `viewer_id` is not a tool argument that the model could invent. `runtime.state` in the tool shows the agent's current state, and `runtime.context` contains the data of the current call.

In [ ]:
@dataclass(frozen=True)
class RequestContext:
    viewer_id: str
    language: str


PROFILES = {"u-101": "Plus", "u-202": "Basic"}
observed = []


@tool
def my_plan(runtime: ToolRuntime[RequestContext]) -> str:
    """Read the user's plan established by the application."""
    # The ID comes from the context. The model does not pass it as a tool argument.
    viewer_id = runtime.context.viewer_id
    # State contains the messages of this conversation; it is not a source of identity.
    observed.append({"viewer_id": viewer_id, "messages_seen": len(runtime.state["messages"])})
    return PROFILES[viewer_id]


## 3. Agent LangChain: input, tool, output

The artificial model will execute the planned tool call and return the planned response. It does not judge for itself which plan is correct. The correct value is returned by Python from `PROFILES`; in a real application, this would be a read from an authorized database. We use `context_schema` as a contract for `context=...`.

In [ ]:
scripted_model = ScriptedModel(responses=[
    AIMessage(content="", tool_calls=[{"name": "my_plan", "args": {}, "id": "call-a"}]),
    AIMessage(content="Plan read from tool."),
    AIMessage(content="", tool_calls=[{"name": "my_plan", "args": {}, "id": "call-b"}]),
    AIMessage(content="Second plan read from tool."),
])
agent = create_agent(
    model=scripted_model,
    tools=[my_plan],
    context_schema=RequestContext,
    checkpointer=InMemorySaver(),
)
thread = {"configurable": {"thread_id": "lesson-09"}}
first = agent.invoke(
    {"messages": [("user", "What is my plan?")]},
    context=RequestContext(viewer_id="u-101", language="en"),
    config=thread,
)
[(m.type, m.content) for m in first["messages"]], observed


`first["messages"]` is the state after the run. It contains the user's message, the model's tool request, the tool result, and the model's response. The artificial model's response does not yet contain the plan name: the text is hardcoded. This deliberately separates the **model's decision** from the **data returned by Python**.

In [ ]:
second = agent.invoke(
    {"messages": [("user", "And now my plan?")]},
    context=RequestContext(viewer_id="u-202", language="en"),
    config=thread,
)
[(m.type, m.content) for m in second["messages"][-4:]], observed[-1]


That `thread_id` preserved the message history, but it **did not freeze the context**: the second turn received a `u-202`. In a production application, do not change the owner of someone else's thread in this manner; this is an experiment demonstrating the API's limits. The thread owner must be verified outside the graph. `InMemorySaver` holds data only for the lifetime of that object in the current process.

In [ ]:
saved = agent.get_state(thread)
len(saved.values["messages"]), [m.content for m in saved.values["messages"] if m.type == "tool"]


## 4. A Closer Look Under the Hood: LangGraph

A LangChain agent operates on a graph. A graph has **nodes** that execute code and a **state** that passes between them. `StateGraph` describes the shape of this state. A node receives the current state and a `Runtime` with context. It returns a **partial update**, and the graph assembles the new state. There is only one node here; we are not covering routing or reducers yet.

In [ ]:
class VisitState(TypedDict):
    question: str
    turns: NotRequired[int]
    answer: NotRequired[str]


def answer_question(state: VisitState, runtime: Runtime[RequestContext]) -> dict:
    # State may change between turns; context is provided for this turn.
    turns = state.get("turns", 0) + 1
    plan = PROFILES[runtime.context.viewer_id]
    return {"turns": turns, "answer": f"Turn {turns}: plan {plan}. Question: {state['question']}"}


builder = StateGraph(VisitState, context_schema=RequestContext)
builder.add_node("answer", answer_question)
builder.add_edge(START, "answer")
builder.add_edge("answer", END)
graph = builder.compile(checkpointer=InMemorySaver())
graph_thread = {"configurable": {"thread_id": "graph-09"}}


In [ ]:
graph_first = graph.invoke(
    {"question": "What is the plan?"},
    context=RequestContext("u-101", "pl"),
    config=graph_thread,
)
graph_second = graph.invoke(
    {"question": "And now?"},
    context=RequestContext("u-202", "en"),
    config=graph_thread,
)
graph_first, graph_second


`turns` increases because the graph reads the previously saved state. `answer` and `question` are state fields. `viewer_id` is used by the node to calculate the answer, but it does not appear as a state field. Without a checkpointer, every call would start from the initial state. A new `thread_id` would also start from the beginning.

In [ ]:
other_thread = graph.invoke(
    {"question": "What is the plan?"},
    context=RequestContext("u-101", "pl"),
    config={"configurable": {"thread_id": "other-graph"}},
)
other_thread


## 5. How to choose the correct location for data?

- **Messages, task progress, tool output:** state. Update via return from a node/hook or `Command(update=...)` from a tool, not through random mutation of `runtime.state`.
- **Identity, permission scope, current request language, database client:** runtime context. Set in the application and provide at every `invoke`. Do not put secrets into messages.
- **Instructions and facts needed by the model right now:** explicitly into the prompt or tool result. `context=...` alone shows nothing to the model.
- **Knowledge from other conversations:** a separate persistent store or database. The state checkpointer of a single conversation is not such a store.

**Exercise:** add a `region` field to `RequestContext`, and a `last_plan` field to `VisitState`. In the node, read the first from `runtime.context`, and update the second via the returned dictionary. Test two threads and two different contexts for the same thread.